# EDA do WebSirene

Inspecao dos dados observacionais do WebSirene organizados por estacao e ano.
Este notebook nao constroi datasets de treinamento nem executa calibracao.

In [ ]:
from __future__ import annotations

import os
from pathlib import Path

import pandas as pd


def find_project_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / '.git').exists():
            return candidate
    raise RuntimeError('Nao foi possivel localizar a raiz do repositorio.')


PROJECT_ROOT = find_project_root(Path.cwd().resolve())
DEFAULT_WEBSIRENE_ROOT = PROJECT_ROOT / 'data' / 'datasets' / 'raw' / 'websirene'
WEBSIRENE_ROOT = Path(os.environ.get('WEBSIRENE_ROOT', DEFAULT_WEBSIRENE_ROOT))

print(f'Projeto: {PROJECT_ROOT}')
print(f'WebSirene: {WEBSIRENE_ROOT}')
if not WEBSIRENE_ROOT.is_dir():
    raise FileNotFoundError(
        'Pasta WebSirene nao encontrada. Defina WEBSIRENE_ROOT para a raiz '
        'com station_id=<id>/year=<ano>/data.parquet.'
    )

## Inventario de estacoes

Cada arquivo anual deve conter, quando disponiveis, `latitude`, `longitude`,
`observation_datetime` e `m15`.

In [ ]:
station_directories = sorted(WEBSIRENE_ROOT.glob('station_id=*'))
print(f'Pastas de estacao: {len(station_directories)}')

records = []
for station_directory in station_directories:
    station_id = int(station_directory.name.removeprefix('station_id='))
    files = sorted(station_directory.glob('year=*/data.parquet'))
    if not files:
        continue

    columns = pd.read_parquet(files[0]).columns
    selected = [column for column in ('latitude', 'longitude') if column in columns]
    location = pd.read_parquet(files[0], columns=selected) if selected else pd.DataFrame()
    record = {'station_id': station_id, 'arquivos_anuais': len(files)}
    for column in selected:
        values = location[column].dropna()
        record[column] = values.iloc[0] if not values.empty else None
    records.append(record)

stations = pd.DataFrame(records).sort_values('station_id').reset_index(drop=True)
display(stations)

## Cobertura temporal e qualidade de m15

In [ ]:
coverage_records = []
for station_directory in station_directories:
    station_id = int(station_directory.name.removeprefix('station_id='))
    for path in sorted(station_directory.glob('year=*/data.parquet')):
        year = int(path.parent.name.removeprefix('year='))
        columns = pd.read_parquet(path).columns
        selected = [column for column in ('observation_datetime', 'm15') if column in columns]
        if not selected:
            continue
        dataframe = pd.read_parquet(path, columns=selected)
        record = {'station_id': station_id, 'ano': year, 'observacoes': len(dataframe)}
        if 'm15' in dataframe:
            record['m15_nulos'] = int(dataframe['m15'].isna().sum())
            record['m15_negativos'] = int((dataframe['m15'] < 0).sum())
            record['m15_maximo'] = dataframe['m15'].max()
        coverage_records.append(record)

coverage = pd.DataFrame(coverage_records).sort_values(['ano', 'station_id'])
print(f'Registros estacao-ano: {len(coverage)}')
display(coverage.head())
display(coverage.groupby('ano')['station_id'].nunique().rename('quantidade_estacoes'))

In [ ]:
summary = coverage.agg(
    estacoes=('station_id', 'nunique'),
    observacoes=('observacoes', 'sum'),
    m15_nulos=('m15_nulos', 'sum'),
    m15_negativos=('m15_negativos', 'sum'),
    m15_maximo=('m15_maximo', 'max'),
)
display(summary.to_frame('valor'))

## Proximos passos

- Use `scripts/analysis/exportar_maximos_websirene.py` para produzir uma
  tabela reprodutivel dos maiores acumulados por ano.
- Mantenha calibracao RGB, treinamento e geracao de datasets em scripts
  ou notebooks especificos, fora desta EDA de fonte.